# conv-stride-downsample — ex1: predict strided conv output length

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-stride-downsample`. Running the final beacon cell reports progress against the `CNN: Stride downsample arithmetic` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Stride downsample arithmetic` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-stride-downsample`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-stride-downsample"
DD_SUBTOPIC = "CNN: Stride downsample arithmetic"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv stride-downsample arithmetic — quick refresher

With kernel size `K`, stride `S`, padding `P = 0`, an input of length `H_in` produces:

```
H_out = (H_in - K) // S + 1
```

**Reading the formula.**
- The first valid window starts at index 0. That accounts for the `+1`.
- Each subsequent window advances by `S` along the input.
- The last window must fit entirely — `H_in - K` is the start index of the *last possible* window; dividing by `S` counts how many strided positions fit in that span.
- Floor division is essential: any partial trailing window is dropped.

**The off-by-one trap.** Naively, doubling the stride should halve the output. But `H_out = H_in // S` is wrong by 1 because of the leading window. For `H_in = 32, K = 3, S = 2`: actual `H_out = (32-3)//2 + 1 = 15`, *not* 16. With same-padding `P = 1` it becomes `(32 + 2 - 3)//2 + 1 = 16`, which IS the clean half — that's why ResNet-style downsampling uses stride-2 *and* same-padding together.

**For `as_strided` windowing.** With stride > 1, the window-index axis advances by `s_w * stride` instead of `s_w`. Skipping is in the *stride*, not in a separate indexing step.

### Exercise 1 — predict strided conv output length

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `H_out = (H_in - K) // S + 1` to predict the output spatial length of a strided 1-D convolution and verify against `F.conv1d` for stride > 1.
> Keywords: stride, downsample, output-shape, off-by-one
> ```

**KCs targeted:** `stride-floor-div-formula`, `stride-leading-window-plus-one`

Implement `ex1_strided_conv_outlen(h_in, k, s)`. Return the output length of a stride-`s` 1-D convolution with kernel size `k` and no padding.

**Formula.**
```
h_out = (h_in - k) // s + 1
```

**Hint.** Use Python integer arithmetic — no tensors. The `+ 1` accounts for the *leading* window starting at index 0; the floor division counts how many additional stride-`s` positions fit before the trailing edge.

**Common off-by-one.** `h_in // s` is the naive 'downsample by s' answer; it is wrong by exactly 1 (it forgets the leading window). The test deliberately exercises this trap.

After your computation, the test cross-checks against a real `F.conv1d` call with random weights, so any off-by-one breaks the assertion immediately.

In [ ]:
def ex1_strided_conv_outlen(h_in: int, k: int, s: int) -> int:
    return (h_in - k) // s + 1


<details><summary>Solution</summary>

```python
def ex1_strided_conv_outlen(h_in: int, k: int, s: int) -> int:
    return (h_in - k) // s + 1
```

**Why the `+ 1` matters.** The number of valid window *positions* is `floor((h_in - k) / s) + 1`. Think of it as: the leading window starts at index 0 (that's 1 position); the floored quotient counts how many additional stride-`s` positions fit before the right edge would push the kernel out of bounds.

**Floor vs round.** PyTorch uses *floor*: a partial trailing window is dropped silently. Some libraries (Caffe, older Theano) used *ceil*. If you ever port code between frameworks, this is the first off-by-one to suspect.

**Same-padding rescue.** With padding `p = (k - 1) // 2` (for odd `k`) and stride `s`, you get `h_out = ceil(h_in / s)` (approximately) — the canonical 'downsample by s' shape that ResNet stride-2 layers rely on. Without padding, the off-by-one bites every time.

**Generalization to 2-D.** Apply the same formula independently to height and width: `(H, W) → ((H-KH)//SH + 1, (W-KW)//SW + 1)`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()